In [2]:
!pip install -q gradio llama-cpp-python huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 11.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.0 MB/s eta 0:00:00


In [3]:
import os
from huggingface_hub import hf_hub_download
from llama_cpp import Llama
import gradio as gr

REPO_ID = "nooruiit-864/qwen2.5-1.5b-base-ai-safety-domain-lora"
GGUF_FILENAME = "model-Q4_K_M.gguf"

print("Downloading GGUF model from Hub...")
model_path = hf_hub_download(repo_id=REPO_ID, filename=GGUF_FILENAME)
print(f"Model downloaded to: {model_path}")

print("Loading model into llama.cpp...")
llm = Llama(model_path=model_path, n_ctx=1024, n_threads=os.cpu_count(), verbose=False)
print("Model loaded. Ready for inference.")


def generate_response(message, history, max_tokens, temperature):
    if not message.strip():
        return "Please enter a prompt."
    output = llm(
        message,
        max_tokens=int(max_tokens),
        temperature=float(temperature),
        stop=["\n\n"],
    )
    return output["choices"][0]["text"].strip()


EXAMPLE_PROMPTS = [
    "Deepfake technology has made it increasingly difficult to",
    "Content authenticity standards such as C2PA are designed to",
    "The biggest risk of generative AI misuse is",
    "Governments are responding to synthetic media by",
    "AI alignment research focuses on",
]

with gr.Blocks(title="Deepfake & AI Safety Domain Model") as demo:
    gr.Markdown(
        """
        # \U0001F6E1\uFE0F AI Safety / Deepfake Misinformation \u2014 Domain-Adapted Qwen2.5-1.5B

        A QLoRA domain-adapted checkpoint of **Qwen2.5-1.5B (base)**, fine-tuned on a
        700-passage corpus covering deepfakes, synthetic media, and AI-related
        misinformation. Merged + quantized (GGUF Q4_K_M) for CPU inference.

        \u26A0\uFE0F **This is a continuation model, not a chat assistant** \u2014 enter a sentence
        prefix and the model will continue it in the domain's style.
        """
    )

    with gr.Row():
        with gr.Column(scale=3):
            prompt_input = gr.Textbox(
                label="Prompt (sentence prefix)",
                placeholder="e.g. Deepfake technology has made it increasingly difficult to",
                lines=3,
            )
            with gr.Row():
                max_tokens_slider = gr.Slider(
                    minimum=20, maximum=200, value=80, step=10, label="Max new tokens"
                )
                temperature_slider = gr.Slider(
                    minimum=0.1, maximum=1.2, value=0.7, step=0.1, label="Temperature"
                )
            generate_btn = gr.Button("Generate", variant="primary")
            output_box = gr.Textbox(label="Model continuation", lines=6)

        with gr.Column(scale=1):
            gr.Markdown("**Try an example prompt:**")
            example_buttons = [gr.Button(p, size="sm") for p in EXAMPLE_PROMPTS]

    generate_btn.click(
        fn=lambda p, mt, t: generate_response(p, None, mt, t),
        inputs=[prompt_input, max_tokens_slider, temperature_slider],
        outputs=output_box,
    )

    for btn, prompt_text in zip(example_buttons, EXAMPLE_PROMPTS):
        btn.click(fn=lambda pt=prompt_text: pt, outputs=prompt_input)

    gr.Markdown(
        """
        ---
        **Base model:** Qwen2.5-1.5B (non-instruct) \u00B7 **Method:** QLoRA (4-bit NF4) \u2192
        merged \u2192 GGUF Q4_K_M \u00B7 **Training corpus:** 700 Wikipedia-sourced passages,
        AI safety / deepfake / misinformation domain \u00B7 Built as part of the
        Planet Beyond AI Engineer Internship (Day 27\u201332).
        """
    )

demo.launch(share=True)

model-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B /  986MB            

model-Q4_K_M.gguf: downloading bytes:           |  0.00B            

Model downloaded to: /root/.cache/huggingface/hub/models--nooruiit-864--qwen2.5-1.5b-base-ai-safety-domain-lora/snapshots/4d25b588f6f21b126ea47923601c8c0fcdefd521/model-Q4_K_M.gguf
Loading model into llama.cpp...
Model loaded. Ready for inference.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e82a00cbc1c189325.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
